In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install rapidfuzz

In [ ]:
"""
Universal Product Categorizer — v5  (Barcode-first direct match, then Embedding+LLM cascade)
=================================================================================================
★★★ NEW IN v5: JSON CHECKPOINT CACHE (crash-safe Stage 3 / Stage 4) ★★★
  Problem: embeddings were already cached to disk (CELL "get_cached_embeddings" → .npy files),
  but Stage 3 (qwen3 LLM adjudication) and Stage 4 (gpt-oss verifier) results only ever lived
  in Python dicts in memory. On Kaggle, if the kernel gets OOM-killed, times out, disconnects,
  or you just need to stop and restart — ALL LLM/verifier work done so far is gone, and a
  re-run pays for every single LLM call again from zero, even for the 95% of rows that were
  already finished and correct.

  Fix: every row's Stage 3 / Stage 4 result is now persisted to a small JSON file on disk
  (llm_stage3_cache.json / verifier_stage4_cache.json inside CACHE_DIR), written after EVERY
  batch completes (not just at the end of the whole run). On the next run:
    1. The cache is loaded first.
    2. Each row gets a stable content-hash key (material_name + candidate categories + hint
       fields it was actually shown, NOT the dataframe row index — index can shift between
       runs, e.g. if the input file changes slightly).
    3. Any row whose key is already in the cache is used directly — zero LLM calls for it.
    4. Only rows missing from the cache get sent to Ollama.
  So a crash at any point loses, at most, the batches that were in flight — never the whole
  run. Genuine failures (batch exhausted all retries with no usable response) are deliberately
  NOT cached, so they get retried automatically on the next run instead of being permanently
  frozen as "UNKNOWN".

  Writes are atomic (write to a temp file + os.replace) so a crash mid-write can never leave
  behind a half-written / corrupted cache file that breaks the *next* run too. If a cache file
  somehow does end up corrupted anyway, loading falls back to starting that cache fresh instead
  of raising and killing the whole pipeline.

  Everything else below (dual-GPU round robin, thinking-mode fix, barcode-first match, source
  filter) is carried over unchanged from v4.
=================================================================================================
"""

# ============================================================================
# %% CELL 0 — CONFIG
# ============================================================================
import os

LULU_XLSX               = "/kaggle/input/datasets/dostmuhammad23/data-ab/categoriesandfinal_brand_standardized (1).xlsx"
LULU_SHEET               = "data"

NEW_FILE_TO_CATEGORIZE  = "/kaggle/input/datasets/dostmuhammad23/data-ab/dim_material_master.csv"
NEW_FILE_SHEET           = 0     # ignored for csv

_stem       = os.path.splitext(os.path.basename(NEW_FILE_TO_CATEGORIZE))[0]
# ★ Kept in its own folder, separate from cache/ and the Ollama logs, so Kaggle's Output
# tab shows it as a single clean file you can download directly. Downloading ONE file
# from Kaggle's output list gives you that file as-is (.xlsx); it's specifically the
# "Download All" button that zips every file under /kaggle/working together — and with
# cache/*.json, cache/*.npy, and ollama_gpu*.log also sitting in that folder, "Download
# All" was always going to hand you a zip no matter what this script does. Putting the
# deliverable in its own subfolder just makes it obvious which single file to grab.
OUTPUT_DIR  = "/kaggle/working/output"
OUTPUT_XLSX = f"{OUTPUT_DIR}/{_stem}_categorized.xlsx"

# ── models (UNCHANGED) ───────────────────────────────────────────────────────
EMBED_MODEL      = "qwen3-embedding:4b"
LLM_MODEL        = "qwen3:14b"
VERIFIER_MODEL         = "gpt-oss:20b"
RUN_VERIFIER_STAGE     = True
VERIFY_BORDERLINE_KNN  = True
VERIFY_KNN_MARGIN      = 0.08

# ── which source_customer values are allowed into the reference corpus ─────
REF_ALLOWED_SOURCE_CUSTOMERS = ["AL_MEERA"]  # GRAND_MALL (4,574 rows) swapped out for
                                                       # AL_MEERA (26,958 rows) — much bigger,
                                                       # more diverse reference corpus for kNN

# ── per-model thinking controls ─────────────────────────────────────────────
# qwen3 (Stage 3) — REVERTED back to thinking disabled. It was briefly turned to "low"
# and tested live: 181 mismatch retries out of 190 batches, and per-batch time exploded
# (ETA went from a few hours to ~22.6 hours remaining). Combined with the exact-count
# JSON schema (minItems==maxItems), the model was consistently running out of its
# num_predict budget on hidden reasoning before it could finish emitting the required
# number of items — exactly the failure mode the original v4 header already documented
# and disabled thinking to avoid. Leaving it off: proven fast and reliable, and the
# other Stage 3 improvements (bigger AL_MEERA reference corpus, hs_code/coo hints, full
# 140-category validation) don't depend on thinking being on.
LLM_THINK               = False   # qwen3 (Stage 3) — thinking OFF (reverted, see above)
VERIFIER_THINK_LEVEL    = "low"   # gpt-oss (Stage 4) — MUST be "low"/"medium"/"high"; bool is ignored

# NOTE: these are now only used as the "shape" for building per-request URLs.
# The actual host (11434 / 11435) is chosen per-request by next_ollama_host()
# further down — see CELL 1B. Do not hardcode a single port anywhere else.
OLLAMA_CHAT_PATH  = "/api/chat"
OLLAMA_EMBED_PATH = "/api/embed"

CACHE_DIR        = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_XLSX), exist_ok=True)

# ── Category acceptance thresholds (unchanged logic) ───────────────────────
KNN_NEIGHBORS       = 5
AUTO_ACCEPT_VOTES   = 4
AUTO_ACCEPT_MIN_SIM = 0.55
LLM_CANDIDATE_TOPN  = 4

# ── Brand fuzzy-match gates ──────────────────────────────────────────────────
BRAND_MIN_LEN_FOR_FUZZY = 4
BRAND_FUZZY_CUTOFF      = 90

# ── ★★★ THROUGHPUT / CONCURRENCY KNOBS ───────────────────────────────────────
# These are BASE (per-GPU) values. They get multiplied by the detected GPU count
# in CELL 1 (see "scaled for N GPU(s)" print) — do not rely on these raw numbers
# after CELL 1 has run, use the scaled globals instead.
LLM_BATCH_SIZE              = 6
_LLM_BATCH_WORKERS_PER_GPU     = 2   # a single T4 running a 14b model chokes above ~2 concurrent
_VERIFIER_BATCH_WORKERS_PER_GPU = 2  # same reasoning for the 20b verifier
_EMBED_MAX_WORKERS_PER_GPU      = 6  # embed calls are cheap, safe to push higher
LLM_MAX_RETRIES     = 3
LLM_REQUEST_TIMEOUT = 600

# Used for the qwen3 (Stage 3) LLM stage only. LLM_THINK=False again (see above), so no
# hidden reasoning tokens compete with it — 700 is comfortable for 6 short JSON items/batch.
OLLAMA_NUM_PREDICT           = 700
# Separate, larger budget for the gpt-oss verifier (Stage 4), since it always spends
# some tokens on its reasoning trace even at "low" and can't be forced to zero.
OLLAMA_NUM_PREDICT_VERIFIER  = 1400
OLLAMA_NUM_CTX      = 8192

EMBED_BATCH_SIZE   = 64

OLLAMA_NUM_PARALLEL      = "2"
OLLAMA_MAX_LOADED_MODELS = "2"
OLLAMA_KEEP_ALIVE        = "60m"

HINT_COLUMNS = [
     "division", "material_desc",
     "mgrp_descr",
     "prdh_descr_1",
     "prdh_descr_2",
     "prdh_descr_3",
 "emgrp_desc", 
    "barcode", "barcode_description",
    # ★ NEW: hs_code is a Harmonized System customs classification code — a strong,
    # largely category-implying signal that was sitting unused in dim_material_master.
    # coo/country (country of origin) were being fed into the embedding text (below) but
    # never actually shown to the LLM as a hint — added here for consistency so Stage 3
    # sees everything the embedding step already does.
    "hs_code", "coo", "country",
]
HINT_TEXT_COLUMNS_FOR_EMBEDDING = [
    "material_desc", "mgrp_descr", "prdh_descr_1", "prdh_descr_2", "prdh_descr_3",
    "emgrp_desc", "barcode_description", "country",
]
BARCODE_COL = "barcode"
NEW_FILE_NAME_COL = None

# ── logging verbosity ───────────────────────────────────────────────────────
# When a batch's parsed item count doesn't match what was sent (should now be rare —
# the exact-count JSON schema constrains this at the grammar level, see CELL 7), the
# code used to dump the FULL raw JSON content + the model's thinking-trace tail to
# stdout every single time. That's invaluable while actively debugging a new failure
# mode, but on a long run it buries the simple "X/Y batches done, ~Zs left" progress
# lines under pages of noise. Leave this False for normal runs — mismatches are still
# counted and shown as a running total next to the progress line — and flip it to True
# only when you're specifically chasing down why batches are failing.
VERBOSE_BATCH_DEBUG = False


# ============================================================================
# %% CELL 1 — INSTALL + START ONE OLLAMA SERVER PER PHYSICAL GPU (auto-detected)
# ============================================================================
import subprocess, time, requests

def _detect_gpu_count() -> int:
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        n = len([l for l in out.stdout.splitlines() if l.strip().startswith("GPU ")])
        return max(n, 1)
    except Exception:
        return 1

_GPU_COUNT = _detect_gpu_count()

subprocess.run("apt-get update -qq && apt-get install -y zstd pciutils -qq", shell=True, check=True)
_GPU_COUNT = _detect_gpu_count()  # re-check now that nvidia-smi/pciutils are guaranteed present

OLLAMA_PORTS = [11434, 11435][:max(_GPU_COUNT, 1)]
OLLAMA_HOSTS = [f"http://127.0.0.1:{p}" for p in OLLAMA_PORTS]
print(f"Detected {_GPU_COUNT} physical GPU(s) → starting {len(OLLAMA_HOSTS)} Ollama server instance(s): {OLLAMA_HOSTS}")
if _GPU_COUNT < 2:
    print("  ⚠ Only 1 GPU detected this session. Code will still run correctly (round-robin "
          "just reuses the single host), but you won't get dual-GPU speedup. If you meant to "
          "get 2 GPUs, check Kaggle Settings > Accelerator / your weekly GPU quota.")

def _ollama_running(host: str) -> bool:
    try:
        return requests.get(f"{host}/api/version", timeout=2).ok
    except Exception:
        return False

if not any(_ollama_running(h) for h in OLLAMA_HOSTS):
    print("Installing Ollama …")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

_OLLAMA_LOG_FILES = {}  # gpu_id -> log path, used for diagnostics if startup fails

def _start_ollama_server(gpu_id: int) -> None:
    """(Re)launch the Ollama server process pinned to one physical GPU. Used both for the
    initial startup below AND later, mid-run, by ensure_ollama_host_alive() (CELL 1B) if a
    server process dies (crash, OOM-kill, driver reset) — restarting is exactly the same
    operation whether it's the first launch or a recovery, so there is only one
    implementation of it. Each restart appends to the SAME per-GPU log file rather than
    overwriting it, so a post-mortem can see every start/crash across the whole session."""
    port = OLLAMA_PORTS[gpu_id]
    host = OLLAMA_HOSTS[gpu_id]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"]     = str(gpu_id)          # pin this instance to ONE physical GPU
    env["OLLAMA_HOST"]              = f"127.0.0.1:{port}"
    env["OLLAMA_NUM_PARALLEL"]      = OLLAMA_NUM_PARALLEL
    env["OLLAMA_MAX_LOADED_MODELS"] = OLLAMA_MAX_LOADED_MODELS
    env["OLLAMA_KEEP_ALIVE"]        = OLLAMA_KEEP_ALIVE
    log_path = f"/kaggle/working/ollama_gpu{gpu_id}.log"
    _OLLAMA_LOG_FILES[gpu_id] = log_path
    print(f"Starting Ollama server on {host} pinned to GPU {gpu_id} "
          f"(NUM_PARALLEL={OLLAMA_NUM_PARALLEL}) → log: {log_path}")
    log_f = open(log_path, "a")   # append — keep history across restarts, don't clobber it
    log_f.write(f"\n\n===== (re)launching at {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n\n")
    log_f.flush()
    subprocess.Popen(["ollama", "serve"], stdout=log_f, stderr=subprocess.STDOUT, env=env)

for gpu_id, (port, host) in enumerate(zip(OLLAMA_PORTS, OLLAMA_HOSTS)):
    if _ollama_running(host):
        print(f"  Ollama already running on {host} (GPU {gpu_id}) — skipping start")
        continue
    _start_ollama_server(gpu_id)

for gpu_id, host in enumerate(OLLAMA_HOSTS):
    for _ in range(30):
        if _ollama_running(host):
            break
        time.sleep(2)
    else:
        tail = ""
        log_path = _OLLAMA_LOG_FILES.get(gpu_id)
        if log_path and os.path.exists(log_path):
            with open(log_path) as f:
                tail = f.read()[-2000:]
        raise RuntimeError(f"Ollama server on {host} did not start in time.\n--- log tail ({log_path}) ---\n{tail}")
print("✓ All Ollama server instance(s) are up")

for host in OLLAMA_HOSTS:
    try:
        r = requests.get(f"{host}/api/ps", timeout=5)
        print(f"  {host} /api/ps:", r.json())
    except Exception as e:
        print(f"  Could not query {host}/api/ps:", e)

print("Run `!nvidia-smi` in a separate cell now — GPU usage should appear only "
      "once Stage 2/3 actually run (idle 0% before that is normal).")

_models_to_pull = [EMBED_MODEL, LLM_MODEL]
if RUN_VERIFIER_STAGE and VERIFIER_MODEL not in _models_to_pull:
    _models_to_pull.append(VERIFIER_MODEL)

pull_env = os.environ.copy()
pull_env["OLLAMA_HOST"] = f"127.0.0.1:{OLLAMA_PORTS[0]}"  # model storage shared across instances — pull once
for _m in _models_to_pull:
    print(f"Pulling model: {_m} (skips if already local) …")
    subprocess.run(["ollama", "pull", _m], check=True, env=pull_env)
print("✓ Ollama ready with all models")


# ============================================================================
# %% CELL 1B — ROUND-ROBIN HOST PICKER + GPU-SCALED CONCURRENCY
# ============================================================================
import itertools, threading

_host_cycle = itertools.cycle(OLLAMA_HOSTS)
_host_lock  = threading.Lock()

def next_ollama_host() -> str:
    with _host_lock:
        return next(_host_cycle)

# ★ CRASH RECOVERY: Ollama server processes can die mid-run — OOM-kill, Kaggle GPU
# driver reset, hitting a session resource limit, etc. When that happens every request
# to that host fails with a bare `Connection refused` (the process is simply gone, not
# just slow), and no amount of HTTP retrying fixes that — the process has to actually be
# relaunched. ensure_ollama_host_alive() is called before every chat request (see
# call_ollama / call_verifier below): if the target host is already up this is a single
# near-instant local port check; if it's down, it (re)launches that GPU's server via
# _start_ollama_server() (CELL 1) and waits for it to come back before letting the
# request through. Thread-safe: if several concurrent batches discover the same dead
# host at once, only the first one actually restarts it — the others just wait.
_ollama_restart_lock = threading.Lock()
_ollama_restart_count = {gpu_id: 0 for gpu_id in range(len(OLLAMA_HOSTS))}

def ensure_ollama_host_alive(host: str, max_wait_s: int = 90) -> bool:
    if _ollama_running(host):
        return True
    with _ollama_restart_lock:
        if _ollama_running(host):  # someone else already restarted it while we waited for the lock
            return True
        gpu_id = OLLAMA_HOSTS.index(host)
        _ollama_restart_count[gpu_id] += 1
        print(f"  ⚠ Ollama on {host} (GPU {gpu_id}) is unreachable (Connection refused — the "
              f"server process died) — restarting it now (restart #{_ollama_restart_count[gpu_id]} "
              f"for this GPU this session) …")
        try:
            _start_ollama_server(gpu_id)
        except Exception as e:
            print(f"  ✗ failed to even launch a new Ollama process on GPU {gpu_id}: {e}")
            return False
        for _ in range(max(max_wait_s // 2, 1)):
            if _ollama_running(host):
                print(f"  ✓ Ollama on {host} (GPU {gpu_id}) is back up and responding")
                return True
            time.sleep(2)
        print(f"  ✗ Ollama on {host} (GPU {gpu_id}) did not come back within {max_wait_s}s — "
              f"check /kaggle/working/ollama_gpu{gpu_id}.log for why it keeps dying "
              f"(common cause: GPU out-of-memory — consider lowering LLM_BATCH_WORKERS / "
              f"VERIFIER_BATCH_WORKERS or OLLAMA_NUM_PARALLEL in CELL 0)")
        return False

# Scale concurrency to however many GPUs we actually got. With 1 GPU this is
# identical to the original tuned values (2 / 2 / 6). With 2 GPUs it doubles,
# so there are enough batches in flight for the round-robin to actually keep
# both GPUs busy instead of just alternating idle time.
LLM_BATCH_WORKERS       = _LLM_BATCH_WORKERS_PER_GPU * _GPU_COUNT
VERIFIER_BATCH_WORKERS  = _VERIFIER_BATCH_WORKERS_PER_GPU * _GPU_COUNT
EMBED_MAX_WORKERS       = _EMBED_MAX_WORKERS_PER_GPU * _GPU_COUNT
print(f"  Concurrency scaled for {_GPU_COUNT} GPU(s): "
      f"LLM_BATCH_WORKERS={LLM_BATCH_WORKERS}, "
      f"VERIFIER_BATCH_WORKERS={VERIFIER_BATCH_WORKERS}, "
      f"EMBED_MAX_WORKERS={EMBED_MAX_WORKERS}")
print(f"  Thinking mode: qwen3 think={LLM_THINK} | gpt-oss think_level={VERIFIER_THINK_LEVEL!r} "
      f"(gpt-oss cannot fully disable thinking)")


# ============================================================================
# %% CELL 1C — ★ JSON CHECKPOINT CACHE (crash-safe Stage 3 / Stage 4) ★
# ============================================================================
import json, hashlib, tempfile

LLM_CACHE_PATH      = os.path.join(CACHE_DIR, "llm_stage3_cache.json")
VERIFIER_CACHE_PATH = os.path.join(CACHE_DIR, "verifier_stage4_cache.json")

_llm_cache_lock      = threading.Lock()
_verifier_cache_lock = threading.Lock()

def _atomic_write_json(path: str, data: dict) -> None:
    """Write JSON to a temp file in the same directory, then os.replace() it over the
    real path. os.replace is atomic on POSIX, so a crash mid-write can never leave a
    half-written / corrupted cache file — the previous good version stays intact until
    the new one is fully flushed to disk, and only then does it become "the" file."""
    d = os.path.dirname(path) or "."
    fd, tmp_path = tempfile.mkstemp(dir=d, prefix=".tmp_ckpt_", suffix=".json")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
    except Exception:
        try:
            os.remove(tmp_path)
        except OSError:
            pass
        raise

def _load_json_cache(path: str, label: str) -> dict:
    if not os.path.exists(path):
        print(f"  [{label}] no existing checkpoint cache found — starting fresh ({path})")
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            print(f"  ✓ [{label}] loaded checkpoint cache: {len(data):,} row(s) already "
                  f"done from a previous run → {path}")
            return data
        print(f"  ⚠ [{label}] checkpoint cache at {path} was not a JSON object — starting fresh")
    except Exception as e:
        # A corrupted / truncated cache (e.g. left over from an old, non-atomic write, or
        # disk got yanked mid-write some other way) must NEVER take down the whole
        # pipeline — worst case we just redo that stage's LLM calls once more.
        print(f"  ⚠ [{label}] checkpoint cache at {path} could not be read ({e!r}) — "
              f"starting fresh instead of crashing")
    return {}

# ─────────────────────────────────────────────────────────────────────────
# ★ KAGGLE CROSS-SESSION RESTORE ★
# /kaggle/working is wiped at the start of every NEW Kaggle session — the JSON files
# above are crash-safe WITHIN one session (kernel dies mid-run -> restart -> resumes),
# but to survive actually CLOSING the notebook and coming back later, Kaggle needs
# something else: an output you committed earlier, re-attached as an input dataset.
#
# Workflow (one-time setup per notebook):
#   1. Run the notebook normally. CELL 1C keeps writing cache/*.json into
#      /kaggle/working/cache/ as it goes — nothing extra to do here.
#   2. Before you stop for the day (or it crashes and you're done for now): click
#      "Save Version" / Commit. This is the ONLY thing that makes /kaggle/working
#      survive past the current session — nothing else does.
#   3. Next session: Notebook menu -> Add Input -> search for THIS notebook's own
#      previous output (or a dataset you made from it) -> attach it. It'll show up
#      under /kaggle/input/<something>/cache/*.json.
#   4. Just re-run the notebook. The scan below finds it automatically (searches
#      every /kaggle/input/*/cache/ and /kaggle/input/*/*/cache/ folder for the two
#      expected filenames) and merges it into /kaggle/working/cache/ before Stage 3/4
#      start — so already-done rows are skipped even across a full session restart,
#      not just a mid-session crash.
#
# First run ever -> nothing to restore -> prints "starting fresh", which is expected.
import glob, shutil

CHECKPOINT_RESTORE_DIRS = []  # add explicit /kaggle/input/.../cache paths here if the
                              # auto-scan below ever misses your attached dataset

def _restore_checkpoint_from_kaggle_inputs() -> None:
    candidates = list(CHECKPOINT_RESTORE_DIRS)
    if os.path.isdir("/kaggle/input"):
        candidates += glob.glob("/kaggle/input/*/cache") + glob.glob("/kaggle/input/*/*/cache")

    restored_any = False
    for cand_dir in candidates:
        for fname in (os.path.basename(LLM_CACHE_PATH), os.path.basename(VERIFIER_CACHE_PATH)):
            src = os.path.join(cand_dir, fname)
            dst = os.path.join(CACHE_DIR, fname)
            if not os.path.exists(src):
                continue
            try:
                with open(src, "r", encoding="utf-8") as f:
                    old = json.load(f)
                if not isinstance(old, dict):
                    continue
            except Exception as e:
                print(f"  ⚠ could not read prior-session cache {src}: {e} — skipping it")
                continue
            if os.path.exists(dst):
                # Merge rather than overwrite: current /kaggle/working copy (if this
                # session already did some work) wins on key collisions, everything
                # restored from the attached dataset is added on top of it.
                try:
                    with open(dst, "r", encoding="utf-8") as f:
                        cur = json.load(f)
                    if not isinstance(cur, dict):
                        cur = {}
                except Exception:
                    cur = {}
                merged = {**old, **cur}
                _atomic_write_json(dst, merged)
                print(f"  ✓ merged {len(old):,} row(s) restored from {src} "
                      f"({len(merged):,} total now in {dst})")
            else:
                shutil.copy2(src, dst)
                print(f"  ✓ restored prior-session checkpoint cache: {src} -> {dst}")
            restored_any = True
    if not restored_any:
        print("  No prior-session checkpoint cache found under /kaggle/input — starting "
              "fresh (expected on the very first run, or if you haven't attached a "
              "committed output as an input dataset yet — see comment above).")

print("Scanning /kaggle/input for a checkpoint cache from a previous (committed) session …")
_restore_checkpoint_from_kaggle_inputs()
# ─────────────────────────────────────────────────────────────────────────

def _safe_float(v, default: float = 0.0) -> float:
    """Coerce a confidence value to a real Python float no matter what shape it arrives
    in. Models occasionally emit confidence as a string (e.g. "0.8" instead of 0.8), or
    it round-trips oddly through the JSON checkpoint cache. Letting even one such value
    into a numeric dataframe column silently upcasts the WHOLE column to object dtype,
    which then blows up later with `TypeError: '<' not supported between str and float`
    the first time Stage 4 tries `work_df["category_conf"] < threshold`. Everything that
    ever gets written into category_conf, or read back out of the checkpoint cache,
    must pass through here first."""
    if v is None:
        return default
    if isinstance(v, bool):  # bool is a subclass of int — guard against True/False leaking in
        return default
    try:
        return float(v)
    except (TypeError, ValueError):
        return default

_llm_cache      = _load_json_cache(LLM_CACHE_PATH, "Stage3 LLM cache")
_verifier_cache = _load_json_cache(VERIFIER_CACHE_PATH, "Stage4 verifier cache")

# ★ MIGRATION: clean up any non-numeric "confidence" values that were already baked into
# the cache file by an earlier run, from before this dtype fix existed. Without this,
# those old bad entries would keep re-poisoning category_conf's dtype forever, even
# though every NEW value written from here on is guaranteed clean — a corrupted cache
# file doesn't heal itself just because the code that writes to it got fixed.
def _sanitize_llm_confidence_cache(cache: dict) -> bool:
    changed = False
    for entry in cache.values():
        if not isinstance(entry, dict) or "confidence" not in entry:
            continue
        raw = entry.get("confidence")
        fixed = _safe_float(raw, 0.0)
        if not isinstance(raw, float) or raw != fixed:
            entry["confidence"] = fixed
            changed = True
    return changed

if _sanitize_llm_confidence_cache(_llm_cache):
    _atomic_write_json(LLM_CACHE_PATH, _llm_cache)
    print(f"  ✓ found and fixed non-numeric confidence value(s) already saved in "
          f"{LLM_CACHE_PATH} from before the dtype fix — cleaned cache written back to disk")

def row_content_key(row: "pd.Series", extra: str = "") -> str:
    """Stable identity for a row's LLM/verifier call, independent of dataframe row index
    (index can shift run-to-run, e.g. if the input file is re-exported slightly
    differently). Hash of exactly the fields that go into the prompt, so if the same
    product with the same candidates/hints was already answered, we reuse that answer;
    if anything about what the model was shown changes, it's treated as a new row and
    re-sent — never silently serving a stale answer for different input."""
    parts = [
        str(row.get("material_name", "")),
        json.dumps(row.get("_cand_categories", []), sort_keys=True, ensure_ascii=False)
            if "_cand_categories" in row.index else "",
        json.dumps(build_hint_block_for_llm(row), sort_keys=True, ensure_ascii=False)
            if "build_hint_block_for_llm" in globals() else "",
        extra,
    ]
    h = hashlib.sha256("||".join(parts).encode("utf-8", errors="ignore")).hexdigest()
    return h[:24]


# ============================================================================
# %% CELL 2 — LOAD DATA
# ============================================================================
import re
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
import concurrent.futures
from rapidfuzz import fuzz, process
from sklearn.neighbors import NearestNeighbors

print("Loading combined workbook …")
kb = pd.read_excel(LULU_XLSX, sheet_name=LULU_SHEET)
kb["material_name"] = kb["material_name"].astype(str)

# The reference corpus (ref_df) is the "data" sheet, restricted to rows whose
# source_customer is LULU or GRANDMALL — any other source_customer is excluded even if
# lulu_category happens to be filled in for that row. This is what we match new items
# against (kNN + LLM candidates), so it must only contain rows we trust.
if "source_customer" not in kb.columns:
    raise KeyError(
        "'source_customer' column not found in the reference workbook — cannot filter to "
        f"{REF_ALLOWED_SOURCE_CUSTOMERS}. Available columns: {list(kb.columns)}"
    )

_source_customer_norm = kb["source_customer"].astype(str).str.strip().str.upper()
_allowed_norm = {s.upper() for s in REF_ALLOWED_SOURCE_CUSTOMERS}

ref_df = kb[
    kb["lulu_category"].notna()
    & (kb["lulu_category"].astype(str).str.strip() != "")
    & _source_customer_norm.isin(_allowed_norm)
].copy()
ref_df = ref_df.dropna(subset=["material_name"]).reset_index(drop=True)

print(f"  source_customer value counts (whole 'data' sheet):")
print(kb["source_customer"].astype(str).str.strip().str.upper().value_counts().to_string())
print(f"  Reference rows kept (source_customer in {REF_ALLOWED_SOURCE_CUSTOMERS} AND "
      f"lulu_category filled): {len(ref_df):,}  (out of {len(kb):,} total rows in the sheet)")

VALID_CATEGORIES = sorted(kb["lulu_category"].dropna().astype(str).str.strip().replace("", pd.NA).dropna().unique().tolist())
print(f"  Valid categories (full taxonomy, whole 'data' sheet — not just the filtered "
      f"reference corpus): {len(VALID_CATEGORIES)}")
_ref_only_categories = sorted(ref_df["lulu_category"].dropna().unique().tolist())
print(f"  Of those, {len(_ref_only_categories)} have at least one example row in the "
      f"filtered reference corpus (source_customer in {REF_ALLOWED_SOURCE_CUSTOMERS}) — "
      f"kNN/LLM candidates can only ever be drawn from this narrower set, since there has "
      f"to be an example product to match against. The other "
      f"{len(VALID_CATEGORIES) - len(_ref_only_categories)} categories from the full "
      f"taxonomy exist as valid output values (won't be wrongly forced to UNKNOWN if ever "
      f"assigned) but can't actually be reached by this pipeline until some reference row "
      f"is added for them.")

brand_series = kb["brand"].dropna().astype(str).str.strip().str.upper()
brand_series = brand_series[brand_series.str.len() > 0]
ALL_BRANDS       = sorted(brand_series.unique().tolist())
FUZZY_BRAND_POOL = [b for b in ALL_BRANDS if len(b) >= BRAND_MIN_LEN_FOR_FUZZY]
BRAND_SET        = set(ALL_BRANDS)
print(f"  Known brands: {len(ALL_BRANDS):,}  (fuzzy pool: {len(FUZZY_BRAND_POOL):,})")

print(f"\nLoading new file to categorize: {NEW_FILE_TO_CATEGORIZE}")
if str(NEW_FILE_TO_CATEGORIZE).lower().endswith((".xlsx", ".xls")):
    new_df = pd.read_excel(NEW_FILE_TO_CATEGORIZE, sheet_name=NEW_FILE_SHEET)
else:
    new_df = pd.read_csv(NEW_FILE_TO_CATEGORIZE)

print(f"  Columns found in new file: {list(new_df.columns)}")
ORIGINAL_COLUMNS = list(new_df.columns)

_NAME_CANDIDATES = ["material_name", "material_desc", "description", "product_name",
                     "item_name", "item_description", "desc", "name"]

if NEW_FILE_NAME_COL and NEW_FILE_NAME_COL in new_df.columns:
    _resolved_name_col = NEW_FILE_NAME_COL
elif NEW_FILE_NAME_COL:
    raise KeyError(
        f"NEW_FILE_NAME_COL was set to '{NEW_FILE_NAME_COL}' but that column isn't in the "
        f"new file. Available columns: {list(new_df.columns)}"
    )
else:
    _resolved_name_col = next((c for c in _NAME_CANDIDATES if c in new_df.columns), None)
    if _resolved_name_col is None:
        raise KeyError(
            "Couldn't find a product-name column in the new file automatically. "
            f"Available columns: {list(new_df.columns)}\n"
            "Set NEW_FILE_NAME_COL in CELL 0 to the correct column name and rerun."
        )

print(f"  Using '{_resolved_name_col}' as the product-name column for this file")

if _resolved_name_col != "material_name":
    new_df["material_name"] = new_df[_resolved_name_col]

n_before = len(new_df)
new_df = new_df.dropna(subset=["material_name"]).reset_index(drop=True)
print(f"  Rows: {n_before:,} -> {len(new_df):,} after dropping blank-name row(s)")


# ============================================================================
# %% CELL 3 — BARCODE-FIRST DIRECT MATCH  (unchanged, still no embedding/LLM cost)
# ============================================================================

def normalize_barcode(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    if isinstance(val, float):
        val = f"{val:.0f}"
    s = str(val).strip()
    if s == "" or s.lower() in ("nan", "none", "0", "0.0"):
        return None
    if "e" in s.lower():
        try:
            s = f"{float(s):.0f}"
        except ValueError:
            pass
    if s.endswith(".0"):
        s = s[:-2]
    s = re.sub(r"\D", "", s)
    if not s or set(s) == {"0"} or len(s) < 6:
        return None
    return s

barcode_to_ref = {}
if BARCODE_COL in ref_df.columns:
    is_lulu_row = (
        ref_df["source_customer"].astype(str).str.upper().eq("LULU")
        if "source_customer" in ref_df.columns else pd.Series([False] * len(ref_df))
    )
    order = ref_df.assign(_is_lulu=is_lulu_row).sort_values("_is_lulu", ascending=False)
    for _, r in order.iterrows():
        bc = normalize_barcode(r[BARCODE_COL])
        if not bc or bc in barcode_to_ref:
            continue
        barcode_to_ref[bc] = {
            "lulu_category": r["lulu_category"],
            "brand": r.get("brand"),
        }
    print(f"  Barcode index built from reference corpus: {len(barcode_to_ref):,} unique barcodes")
else:
    print(f"  WARNING: '{BARCODE_COL}' not found in reference corpus — barcode-first match disabled")

if BARCODE_COL in new_df.columns:
    new_df["_barcode_norm"] = new_df[BARCODE_COL].apply(normalize_barcode)
else:
    new_df["_barcode_norm"] = None

new_df["matched_via_barcode"] = new_df["_barcode_norm"].apply(lambda b: bool(b and b in barcode_to_ref))

barcode_matched_mask = new_df["matched_via_barcode"]
n_bc = int(barcode_matched_mask.sum())
print(f"  ★ Direct barcode matches: {n_bc:,} / {len(new_df):,} rows — these SKIP embedding + LLM entirely")

needs_categorization_mask = ~barcode_matched_mask


# ============================================================================
# %% CELL 4 — HINT-COLUMN TEXT BUILDERS  (unchanged)
# ============================================================================

def _clean_val(v) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    return "" if s.lower() in ("nan", "none", "") else s

def build_hint_text_for_embedding(row: pd.Series) -> str:
    parts = []
    for col in HINT_TEXT_COLUMNS_FOR_EMBEDDING:
        if col in row.index:
            v = _clean_val(row[col])
            if v:
                parts.append(v)
    return " | ".join(parts)

def build_hint_block_for_llm(row: pd.Series) -> dict:
    block = {}
    for col in HINT_COLUMNS:
        if col in row.index:
            v = _clean_val(row[col])
            if v:
                block[col] = v
    return block

new_df["_embed_text"] = (
    new_df["material_name"].astype(str) + " | " + new_df.apply(build_hint_text_for_embedding, axis=1)
).str.strip(" |")

if any(c in ref_df.columns for c in HINT_TEXT_COLUMNS_FOR_EMBEDDING):
    ref_df["_embed_text"] = (
        ref_df["material_name"].astype(str) + " | " + ref_df.apply(build_hint_text_for_embedding, axis=1)
    ).str.strip(" |")
else:
    ref_df["_embed_text"] = ref_df["material_name"].astype(str)


# ============================================================================
# %% CELL 5 — STAGE 1: RULE-BASED BRAND EXTRACTION  (unchanged, CPU-only, already fast)
# ============================================================================

def guess_brand_rule(name: str):
    toks = re.findall(r"[A-Z0-9']+", str(name).upper())
    for n in (3, 2, 1):
        if len(toks) >= n:
            cand = " ".join(toks[:n])
            if cand in BRAND_SET:
                return cand, "rule_exact", 100.0
    cand = " ".join(toks[:2]) if len(toks) >= 2 else (toks[0] if toks else "")
    if len(cand) < 5:
        return None, "rule_none", 0.0
    match = process.extractOne(cand, FUZZY_BRAND_POOL, scorer=fuzz.token_sort_ratio,
                                score_cutoff=BRAND_FUZZY_CUTOFF)
    if match:
        return match[0], "rule_fuzzy", float(match[1])
    return None, "rule_none", 0.0

new_df["brand_final"]  = new_df.get("brand")
new_df["brand_method"] = np.where(new_df.get("brand", pd.Series([None]*len(new_df))).notna()
                                   & (new_df.get("brand", pd.Series([""]*len(new_df))).astype(str).str.strip() != ""),
                                   "given", "")

rule_targets = needs_categorization_mask & (
    new_df["brand_final"].isna() | (new_df["brand_final"].astype(str).str.strip() == "")
)
print(f"Stage 1 — rule-based brand extraction on {int(rule_targets.sum()):,} non-barcode row(s) …")
guesses = new_df.loc[rule_targets, "material_name"].apply(guess_brand_rule)
new_df.loc[rule_targets, "brand_final"]  = [g[0] for g in guesses]
new_df.loc[rule_targets, "brand_method"] = [g[1] for g in guesses]
print(new_df.loc[needs_categorization_mask, "brand_method"].value_counts().to_string())


# ============================================================================
# %% CELL 6 — EMBEDDING HELPERS — round-robin across GPUs  (unchanged)
# ============================================================================

def _hash_list(items) -> str:
    h = hashlib.sha256()
    for it in items:
        h.update(str(it).encode("utf-8", errors="ignore")); h.update(b"\x00")
    return h.hexdigest()[:20]

def _ollama_embed_chunk(chunk: list, model: str) -> list:
    payload = {"model": model, "input": chunk}
    last_err = None
    for attempt in range(3):
        host = next_ollama_host()            # round-robin: alternates GPU 0 / GPU 1
        if not ensure_ollama_host_alive(host):
            last_err = f"{host}: server down and could not be restarted"; time.sleep(1 * (attempt + 1)); continue
        embed_url = f"{host}{OLLAMA_EMBED_PATH}"
        try:
            resp = requests.post(embed_url, json=payload, timeout=120)
            resp.raise_for_status()
            embs = resp.json().get("embeddings")
            if not embs or len(embs) != len(chunk):
                raise ValueError("unexpected /api/embed response shape")
            return embs
        except Exception as e:
            last_err = f"{host}: {e}"; time.sleep(1 * (attempt + 1))
    raise RuntimeError(f"Ollama embedding failed: {last_err}")

def ollama_embed_texts(texts: list, model: str, batch_size: int = EMBED_BATCH_SIZE,
                        max_workers: int = None, label: str = "") -> np.ndarray:
    if max_workers is None:
        max_workers = EMBED_MAX_WORKERS   # resolved at call time, already GPU-scaled
    if not texts:
        return np.zeros((0, 0), dtype=np.float32)
    n_chunks = (len(texts) + batch_size - 1) // batch_size
    chunks = [texts[i*batch_size:(i+1)*batch_size] for i in range(n_chunks)]
    results = [None] * n_chunks
    t0 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_ollama_embed_chunk, c, model): i for i, c in enumerate(chunks)}
        done = 0
        for fut in concurrent.futures.as_completed(futs):
            results[futs[fut]] = fut.result()
            done += 1
            elapsed = time.time() - t0
            eta = (elapsed / done) * (n_chunks - done)
            if done % 5 == 0 or done == n_chunks:
                print(f"  [{label}] embedding chunk {done}/{n_chunks}  "
                      f"({elapsed:.1f}s elapsed, ~{eta:.1f}s left)")
    flat = [v for chunk in results for v in chunk]
    arr = np.array(flat, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True); norms[norms == 0] = 1.0
    return arr / norms

def get_cached_embeddings(texts: list, model: str, label: str) -> np.ndarray:
    if not texts:
        return np.zeros((0, 0), dtype=np.float32)
    key = _hash_list(texts) + "_" + model.replace(":", "_").replace("/", "_")
    path = os.path.join(CACHE_DIR, f"emb_{label}_{key}.npy")
    if os.path.exists(path):
        try:
            arr = np.load(path)
            if arr.shape[0] == len(texts):
                print(f"  ✓ [{label}] embedding cache hit → {path}")
                return arr
        except Exception:
            pass
    print(f"  Encoding {len(texts):,} texts via Ollama (model={model}) …")
    arr = ollama_embed_texts(texts, model=model, label=label)
    try:
        np.save(path, arr)
    except Exception as e:
        print(f"  ⚠ could not cache embeddings: {e}")
    return arr


# ============================================================================
# %% CELL 7 — STAGE 2: EMBEDDING kNN  +  STAGE 3: LLM ADJUDICATION
#              (★ now with crash-safe JSON checkpoint cache ★)
# ============================================================================

work_df = new_df.loc[needs_categorization_mask].copy()
print(f"Stage 2 — embedding kNN for {len(work_df):,} row(s) without a barcode match …")

ref_emb  = get_cached_embeddings(ref_df["_embed_text"].tolist(), EMBED_MODEL, "reference")
work_emb = get_cached_embeddings(work_df["_embed_text"].tolist(), EMBED_MODEL, "newfile_nobarcode")

cand_categories, cand_examples, auto_category, auto_confidence = [], [], [], []

if len(work_df) and ref_emb.shape[0]:
    nn = NearestNeighbors(n_neighbors=min(KNN_NEIGHBORS, len(ref_df)), metric="cosine").fit(ref_emb)
    dist, idx = nn.kneighbors(work_emb)
    sims = 1 - dist
    ref_cats  = ref_df["lulu_category"].tolist()
    ref_names = ref_df["material_name"].tolist()

    for i in range(len(work_df)):
        row_cats = [ref_cats[j] for j in idx[i]]
        row_sims = sims[i]
        c = Counter(row_cats)
        top_cat, votes = c.most_common(1)[0]
        avg_sim = float(np.mean([s for s, cat in zip(row_sims, row_cats) if cat == top_cat]))

        seen, cands, ex = set(), [], {}
        order = sorted(range(len(row_cats)), key=lambda k: -row_sims[k])
        for k in order:
            cat = row_cats[k]
            if cat in seen:
                continue
            seen.add(cat); cands.append(cat); ex[cat] = ref_names[idx[i][k]]
            if len(cands) >= LLM_CANDIDATE_TOPN:
                break
        cand_categories.append(cands)
        cand_examples.append(ex)

        if votes >= AUTO_ACCEPT_VOTES and avg_sim >= AUTO_ACCEPT_MIN_SIM:
            auto_category.append(top_cat); auto_confidence.append(round(avg_sim, 3))
        else:
            auto_category.append(None); auto_confidence.append(round(avg_sim, 3))

work_df["_cand_categories"] = cand_categories
work_df["_cand_examples"]   = cand_examples
work_df["category_auto"]    = auto_category
work_df["category_conf"]    = auto_confidence

n_auto = work_df["category_auto"].notna().sum() if len(work_df) else 0
print(f"  Auto-accepted category (embedding kNN, no LLM): {n_auto:,} / {len(work_df):,} "
      f"({(n_auto/len(work_df)*100) if len(work_df) else 0:.1f}%)")

# ── STAGE 3: LLM (prompt unchanged) ─────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a product taxonomist for a Qatar FMCG hypermarket (LULU). Map a retail "
    "product to the single best category from the provided candidate list belonging "
    "to the LULU master taxonomy. Judge by the product's true semantic meaning, not "
    "by matching words literally. Beware misleading tokens: 'Rice Flour' is a FLOUR "
    "not a RICE; 'Chicken Masala' is a spice mix, not fresh chicken; 'Diet Coke' is a "
    "soft drink, not a diet/health product. Use any hint fields provided (material "
    "description, merchandise group, product hierarchy descriptions, country of "
    "origin, barcode description) as supporting context, and treat the customer's own "
    "category as only a WEAK hint — it may be wrong or inconsistent with LULU's "
    "taxonomy. You MUST choose exactly one category from the candidate list given for "
    "each item; if truly none fit, and only then, answer \"UNKNOWN\". Also propose a "
    "brand guess only where asked (need_brand=true) and only if a real brand name is "
    "identifiable from the text — never invent one.\n\n"
    "Respond with ONLY a single JSON OBJECT (no prose, no markdown fences) in this "
    "exact shape:\n"
    '{"items": [{"category": "<exact candidate or UNKNOWN>", "brand": "...", '
    '"confidence": 0.0, "reason": "<=12 words"}, ...]}\n'
    "The \"items\" array must contain exactly one object per input item, in the SAME "
    "ORDER as given."
)

def build_batch_payload(rows: pd.DataFrame):
    items = []
    for _, r in rows.iterrows():
        need_brand = str(r.get("brand_final", "")).strip() in ("", "nan", "None")
        cands = r["_cand_categories"]
        ex    = r["_cand_examples"]
        items.append({
            "name": r["material_name"],
            "customer_category_hint": _clean_val(r.get("customer_category")) or None,
            "hints": build_hint_block_for_llm(r),
            "candidates": [{"category": c, "example": ex[c]} for c in cands],
            "need_brand": bool(need_brand),
        })
    return items

_debug_counts_lock    = threading.Lock()
_llm_mismatch_count      = 0   # how many Stage 3 batches returned the wrong item count
_verifier_mismatch_count = 0   # how many Stage 4 batches returned the wrong item count

def call_ollama(items: list, debug_label: str = ""):
    host = next_ollama_host()            # round-robin: alternates GPU 0 / GPU 1
    if not ensure_ollama_host_alive(host):
        raise RuntimeError(f"Ollama on {host} is down and could not be restarted")
    chat_url = f"{host}{OLLAMA_CHAT_PATH}"
    user_msg = "Classify these products:\n" + json.dumps({"items": items}, ensure_ascii=False)
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "format": _schema_for_exact_count(LLM_RESPONSE_SCHEMA_TEMPLATE, len(items)),
        "think": LLM_THINK,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": OLLAMA_NUM_PREDICT, "num_ctx": OLLAMA_NUM_CTX},
    }
    resp = requests.post(chat_url, json=payload, timeout=LLM_REQUEST_TIMEOUT)
    resp.raise_for_status()
    message = resp.json()["message"]
    content = message.get("content", "")  # .get(), not [..] — content can legitimately be absent/empty
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        matches = re.findall(r"\{[^{}]*\}", content)
        parsed = []
        for m in matches:
            try:
                obj = json.loads(m)
                if "category" in obj or "brand" in obj:
                    parsed.append(obj)
            except json.JSONDecodeError:
                break
    if isinstance(parsed, dict):
        if "items" in parsed and isinstance(parsed["items"], list):
            parsed = parsed["items"]
        else:
            for v in parsed.values():
                if isinstance(v, list):
                    parsed = v; break
            else:
                parsed = [parsed] if parsed else []
    if not isinstance(parsed, list):
        raise ValueError("Model did not return a JSON array")
    if len(parsed) != len(items):
        global _llm_mismatch_count
        with _debug_counts_lock:
            _llm_mismatch_count += 1
        if VERBOSE_BATCH_DEBUG:
            print(f"  [debug {debug_label} host={host}] raw content (first 600 chars): {content[:600]!r}")
            thinking = message.get("thinking") or ""
            if thinking:
                print(f"  [debug {debug_label} host={host}] thinking field had {len(thinking)} chars "
                      f"(num_predict={OLLAMA_NUM_PREDICT}, think={LLM_THINK}) — model may have run out "
                      f"of budget before answering. Tail: {thinking[-300:]!r}")
    return parsed

_LLM_NO_RESPONSE_SENTINEL = "no LLM response"  # never cached — always eligible for retry next run

# ★ STRUCTURED OUTPUT SCHEMAS — grammar-constrained decoding, not just a JSON hint ★
# Passing "format": "json" only tells the model "please output JSON" as a soft
# instruction — the model can and occasionally does still emit syntactically broken
# output (missing braces/colons mid-array), especially on longer batches, which then
# fails json.loads(), falls through to the regex fallback, and burns a retry. Ollama's
# structured-outputs feature accepts an actual JSON SCHEMA (not just the string "json")
# as the "format" value and compiles it into a grammar that constrains token-by-token
# decoding — the model is literally unable to produce syntactically invalid JSON or a
# field of the wrong type. This doesn't guarantee the array LENGTH always matches the
# input (that's still a semantic choice by the model), but it eliminates the "corrupted
# mid-object" failure mode seen in the logs entirely.
LLM_RESPONSE_SCHEMA_TEMPLATE = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "category":   {"type": "string"},
                    "brand":      {"type": "string"},
                    "confidence": {"type": "number"},
                    "reason":     {"type": "string"},
                },
                "required": ["category", "confidence", "reason"],
            },
        }
    },
    "required": ["items"],
}

VERIFIER_RESPONSE_SCHEMA_TEMPLATE = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "verified":   {"type": "boolean"},
                    "ambiguous":  {"type": "boolean"},
                    "confidence": {"type": "number"},
                    "note":       {"type": "string"},
                },
                "required": ["verified", "ambiguous", "confidence", "note"],
            },
        }
    },
    "required": ["items"],
}

def _schema_for_exact_count(template: dict, n: int) -> dict:
    """Clone a schema template with minItems == maxItems == n baked into the 'items'
    array. Passing "format":"json" only asks nicely for valid syntax — even with a full
    JSON-schema grammar constraining every field's type, an unconstrained array can still
    legally have any length, and gpt-oss was observed doing exactly that: consistently
    returning 5 well-formed items for a 6-item batch (not corrupted, just under-counted —
    e.g. collapsing two very similar "whole fish" items into one judgement). Forcing
    minItems/maxItems to the exact input count removes that degree of freedom entirely:
    the grammar will not allow the model to close the array early or pad it. Rebuilt
    per-batch (not a single shared global) because batch size varies — 6 for most
    batches, fewer for the last partial batch of a stage."""
    s = json.loads(json.dumps(template))  # cheap deep copy, avoids importing copy module
    s["properties"]["items"]["minItems"] = n
    s["properties"]["items"]["maxItems"] = n
    return s

def _process_llm_batch(batch_rows_idx, batch_df):
    """Runs one Stage-3 batch end to end (with retries). Safe to call from multiple threads."""
    items = build_batch_payload(batch_df)
    result = None
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            candidate = call_ollama(items, debug_label=f"{batch_rows_idx[0]} attempt {attempt}")
            if len(candidate) == len(items):
                result = candidate; break
            if attempt == LLM_MAX_RETRIES and candidate:
                result = candidate
        except Exception as e:
            print(f"  ⚠ batch starting {batch_rows_idx[0]} attempt {attempt} failed: {e}")
        time.sleep(2)

    out = {}
    if not result:
        for ridx in batch_rows_idx:
            out[ridx] = {"category": "UNKNOWN", "confidence": 0.0, "reason": _LLM_NO_RESPONSE_SENTINEL}
        return out

    if len(result) < len(batch_rows_idx):
        result = result + [{}] * (len(batch_rows_idx) - len(result))
    for ridx, o in zip(batch_rows_idx, result):
        out[ridx] = o
    return out

needs_llm_mask_local = work_df["category_auto"].isna() if len(work_df) else pd.Series([], dtype=bool)
llm_idx = work_df.index[needs_llm_mask_local].tolist() if len(work_df) else []

# ★ CHECKPOINT: compute each row's stable content-hash key BEFORE deciding what to send —
# anything whose key is already in _llm_cache (from a previous, possibly crashed run) gets
# reused for free instead of being re-sent to Ollama.
work_df["_llm_cache_key"] = ""
if llm_idx:
    work_df.loc[llm_idx, "_llm_cache_key"] = work_df.loc[llm_idx].apply(row_content_key, axis=1)

llm_idx_cached = [i for i in llm_idx if work_df.at[i, "_llm_cache_key"] in _llm_cache]
llm_idx_to_run = [i for i in llm_idx if work_df.at[i, "_llm_cache_key"] not in _llm_cache]

print(f"Stage 3 — {len(llm_idx):,} row(s) need LLM categorization: "
      f"{len(llm_idx_cached):,} already finished in checkpoint cache (free), "
      f"{len(llm_idx_to_run):,} to send to Ollama "
      f"({LLM_MODEL}, think={LLM_THINK}) — batch_size={LLM_BATCH_SIZE}, "
      f"{LLM_BATCH_WORKERS} batches in parallel, round-robin across {len(OLLAMA_HOSTS)} GPU host(s) …")

llm_category, llm_brand, llm_confidence, llm_reason = {}, {}, {}, {}

# Pull already-finished rows straight out of the checkpoint cache — zero LLM calls, zero risk.
for ridx in llm_idx_cached:
    cached = _llm_cache[work_df.at[ridx, "_llm_cache_key"]]
    llm_category[ridx]   = cached.get("category", "UNKNOWN")
    llm_confidence[ridx] = _safe_float(cached.get("confidence"), 0.0)
    llm_reason[ridx]     = cached.get("reason", "")
    if cached.get("brand"):
        llm_brand[ridx] = cached["brand"]

batches = []
for batch_start in range(0, len(llm_idx_to_run), LLM_BATCH_SIZE):
    batch_rows_idx = llm_idx_to_run[batch_start: batch_start + LLM_BATCH_SIZE]
    batches.append((batch_rows_idx, work_df.loc[batch_rows_idx]))

t0 = time.time()
done_batches = 0
if batches:
    with concurrent.futures.ThreadPoolExecutor(max_workers=LLM_BATCH_WORKERS) as ex:
        futs = {ex.submit(_process_llm_batch, b_idx, b_df): b_idx for b_idx, b_df in batches}
        for fut in concurrent.futures.as_completed(futs):
            result_map = fut.result()
            for ridx, out in result_map.items():
                cat = str(out.get("category", "UNKNOWN")).strip() or "UNKNOWN"
                if cat != "UNKNOWN" and cat not in VALID_CATEGORIES:
                    cat = "UNKNOWN"
                reason = str(out.get("reason", ""))[:120]
                llm_category[ridx]   = cat
                llm_confidence[ridx] = _safe_float(out.get("confidence"), 0.0)  # never let a str/None through
                llm_reason[ridx]     = reason
                brand_guess = str(out.get("brand", "")).strip()
                if brand_guess and brand_guess.lower() not in ("none", "nan", "unknown"):
                    llm_brand[ridx] = brand_guess.upper()

                # ★ CHECKPOINT: persist this row's result in memory immediately. Genuine
                # failures (no usable LLM response after all retries) are deliberately NOT
                # cached, so they're retried fresh on the next run instead of being
                # permanently frozen as "UNKNOWN".
                if reason != _LLM_NO_RESPONSE_SENTINEL:
                    key = work_df.at[ridx, "_llm_cache_key"]
                    with _llm_cache_lock:
                        _llm_cache[key] = {
                            "category": cat,
                            "confidence": llm_confidence[ridx],
                            "reason": reason,
                            "brand": llm_brand.get(ridx),
                        }
            done_batches += 1
            # ★ CHECKPOINT: flush the whole cache to disk after EVERY batch. The file is
            # small (a few hundred bytes per row) so this is cheap, and it guarantees that
            # at most one batch's worth of LLM calls is ever redone after a crash.
            with _llm_cache_lock:
                _atomic_write_json(LLM_CACHE_PATH, _llm_cache)
            if done_batches % 10 == 0 or done_batches == len(batches):
                elapsed = time.time() - t0
                eta = (elapsed / done_batches) * (len(batches) - done_batches)
                mismatch_note = f", {_llm_mismatch_count} mismatch retr" \
                                f"{'y' if _llm_mismatch_count == 1 else 'ies'} so far" \
                                if _llm_mismatch_count else ""
                print(f"  … {done_batches}/{len(batches)} batches done "
                      f"({elapsed:.0f}s elapsed, ~{eta:.0f}s left{mismatch_note}) — "
                      f"checkpoint saved → {LLM_CACHE_PATH}")
else:
    print("  Nothing to send — every row was already in the checkpoint cache.")

if _llm_mismatch_count:
    print(f"  ⚠ Stage 3 total: {_llm_mismatch_count} batch attempt(s) returned the wrong "
          f"item count and had to retry (set VERBOSE_BATCH_DEBUG=True in CELL 0 to see the "
          f"raw content for these next time)")
print("✓ Stage 3 complete")

work_df["lulu_category_final"] = work_df["category_auto"]
work_df["category_method"]     = np.where(work_df["category_auto"].notna(), "embed_knn_auto", "")
work_df["llm_reason"]          = ""

for ridx, cat in llm_category.items():
    work_df.at[ridx, "lulu_category_final"] = cat
    work_df.at[ridx, "category_method"] = "llm"
for ridx, conf in llm_confidence.items():
    if work_df.at[ridx, "category_method"] == "llm":
        work_df.at[ridx, "category_conf"] = _safe_float(conf, 0.0)
for ridx, r in llm_reason.items():
    work_df.at[ridx, "llm_reason"] = r
for ridx, b in llm_brand.items():
    cur = work_df.at[ridx, "brand_final"]
    if pd.isna(cur) or str(cur).strip() == "":
        work_df.at[ridx, "brand_final"]  = b
        work_df.at[ridx, "brand_method"] = "llm"

# ★ Final safety net: no matter how a stray string/None slipped in above, force this
# column back to a real numeric dtype before Stage 4 does any `<` comparison on it.
# Without this, ONE bad value silently upcasts the whole column to `object` dtype and
# the comparison raises `TypeError: '<' not supported between instances of 'str' and
# 'float'` — this is exactly what happened before this fix.
work_df["category_conf"] = pd.to_numeric(work_df["category_conf"], errors="coerce").fillna(0.0)


# ============================================================================
# %% CELL 7B — STAGE 4: SECOND-OPINION VERIFIER LLM
#               (★ now with crash-safe JSON checkpoint cache ★)
# ============================================================================

VERIFIER_SYSTEM_PROMPT = (
    "You are a strict QA reviewer for a product-categorization system. You will be "
    "shown a product, the category it was assigned, and a reference example product "
    "that already belongs to that category. Decide if the assignment is CORRECT, or "
    "AMBIGUOUS/WRONG. Judge by true semantic meaning, not superficial word overlap — "
    "the same misleading-token traps apply (e.g. a flavored spice mix named after an "
    "ingredient is not that raw ingredient; a diet/zero soft drink is still a "
    "beverage, not a health product). If the product could plausibly belong to more "
    "than one very different category, or the assigned category is a poor semantic "
    "fit, mark it ambiguous/wrong rather than rubber-stamping it.\n\n"
    "Respond with ONLY a single JSON OBJECT (no prose, no markdown fences):\n"
    '{"items": [{"verified": true/false, "ambiguous": true/false, '
    '"confidence": 0.0, "note": "<=15 words"}, ...]}\n'
    "One object per input item, SAME ORDER as given."
)

def build_verifier_payload(rows: pd.DataFrame):
    items = []
    for _, r in rows.iterrows():
        assigned_cat = r["lulu_category_final"]
        ex = r["_cand_examples"].get(assigned_cat) if isinstance(r.get("_cand_examples"), dict) else None
        items.append({
            "name": r["material_name"],
            "assigned_category": assigned_cat,
            "reference_example": ex,
            "hints": build_hint_block_for_llm(r),
        })
    return items

def call_verifier(items: list, debug_label: str = ""):
    host = next_ollama_host()            # round-robin: alternates GPU 0 / GPU 1
    if not ensure_ollama_host_alive(host):
        raise RuntimeError(f"Ollama on {host} is down and could not be restarted")
    chat_url = f"{host}{OLLAMA_CHAT_PATH}"
    user_msg = "Review these category assignments:\n" + json.dumps({"items": items}, ensure_ascii=False)
    payload = {
        "model": VERIFIER_MODEL,
        "messages": [
            {"role": "system", "content": VERIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "format": _schema_for_exact_count(VERIFIER_RESPONSE_SCHEMA_TEMPLATE, len(items)),
        "think": VERIFIER_THINK_LEVEL,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": OLLAMA_NUM_PREDICT_VERIFIER, "num_ctx": OLLAMA_NUM_CTX},
    }
    resp = requests.post(chat_url, json=payload, timeout=LLM_REQUEST_TIMEOUT)
    resp.raise_for_status()
    message = resp.json()["message"]
    content = message.get("content", "")
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        matches = re.findall(r"\{[^{}]*\}", content)
        parsed = []
        for m in matches:
            try:
                obj = json.loads(m)
                if "verified" in obj:
                    parsed.append(obj)
            except json.JSONDecodeError:
                break
    if isinstance(parsed, dict):
        parsed = parsed.get("items", [parsed] if parsed else [])
    if not isinstance(parsed, list):
        raise ValueError("Verifier did not return a JSON array")
    if len(parsed) != len(items):
        global _verifier_mismatch_count
        with _debug_counts_lock:
            _verifier_mismatch_count += 1
        if VERBOSE_BATCH_DEBUG:
            print(f"  [verifier debug {debug_label} host={host}] raw (first 500 chars): {content[:500]!r}")
            thinking = message.get("thinking") or ""
            if thinking:
                print(f"  [verifier debug {debug_label} host={host}] thinking field had {len(thinking)} "
                      f"chars (num_predict={OLLAMA_NUM_PREDICT_VERIFIER}, think={VERIFIER_THINK_LEVEL!r}) "
                      f"— gpt-oss may still be running long on its trace. Tail: {thinking[-300:]!r}")
    return parsed

def _process_verifier_batch(batch_rows_idx, batch_df):
    items = build_verifier_payload(batch_df)
    result = None
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            candidate = call_verifier(items, debug_label=f"{batch_rows_idx[0]} attempt {attempt}")
            if len(candidate) == len(items):
                result = candidate; break
        except Exception as e:
            print(f"  ⚠ verifier batch starting {batch_rows_idx[0]} attempt {attempt} failed: {e}")
        time.sleep(2)

    out = {}
    if not result:
        for ridx in batch_rows_idx:
            out[ridx] = None  # signal: leave original verdict untouched, and never cache this
        return out
    if len(result) < len(batch_rows_idx):
        result = result + [{}] * (len(batch_rows_idx) - len(result))
    for ridx, o in zip(batch_rows_idx, result):
        out[ridx] = o
    return out

work_df["verified"]        = None
work_df["verifier_ambiguous"] = None
work_df["verifier_note"]      = ""
work_df["verifier_model"]     = ""

if RUN_VERIFIER_STAGE and len(work_df):
    verify_mask = (work_df["category_method"] == "llm") & (work_df["lulu_category_final"] != "UNKNOWN")
    if VERIFY_BORDERLINE_KNN:
        borderline = (work_df["category_method"] == "embed_knn_auto") & \
                     (work_df["category_conf"] < AUTO_ACCEPT_MIN_SIM + VERIFY_KNN_MARGIN)
        verify_mask = verify_mask | borderline

    verify_idx = work_df.index[verify_mask].tolist()

    # ★ CHECKPOINT: key includes the ASSIGNED category (via extra=) — if Stage 3 assigns a
    # different category on a re-run, that's treated as a new verification question rather
    # than silently reusing a verdict for a different assignment.
    work_df["_verifier_cache_key"] = ""
    if verify_idx:
        work_df.loc[verify_idx, "_verifier_cache_key"] = work_df.loc[verify_idx].apply(
            lambda r: row_content_key(r, extra=f"assigned={r['lulu_category_final']}"), axis=1
        )

    verify_idx_cached = [i for i in verify_idx if work_df.at[i, "_verifier_cache_key"] in _verifier_cache]
    verify_idx_to_run = [i for i in verify_idx if work_df.at[i, "_verifier_cache_key"] not in _verifier_cache]

    print(f"Stage 4 — {len(verify_idx):,} assignment(s) need verification: "
          f"{len(verify_idx_cached):,} already finished in checkpoint cache (free), "
          f"{len(verify_idx_to_run):,} to send to verifier "
          f"({VERIFIER_MODEL}, think={VERIFIER_THINK_LEVEL!r}) — "
          f"batch_size={LLM_BATCH_SIZE}, {VERIFIER_BATCH_WORKERS} batches in parallel, "
          f"round-robin across {len(OLLAMA_HOSTS)} GPU host(s) …")

    # Pull already-finished verdicts straight out of the checkpoint cache.
    for ridx in verify_idx_cached:
        cached = _verifier_cache[work_df.at[ridx, "_verifier_cache_key"]]
        work_df.at[ridx, "verified"]           = cached.get("verified")
        work_df.at[ridx, "verifier_ambiguous"] = cached.get("ambiguous")
        work_df.at[ridx, "verifier_note"]      = cached.get("note", "")
        work_df.at[ridx, "verifier_model"]     = VERIFIER_MODEL

    v_batches = []
    for batch_start in range(0, len(verify_idx_to_run), LLM_BATCH_SIZE):
        batch_rows_idx = verify_idx_to_run[batch_start: batch_start + LLM_BATCH_SIZE]
        v_batches.append((batch_rows_idx, work_df.loc[batch_rows_idx]))

    t0 = time.time()
    done_batches = 0
    if v_batches:
        with concurrent.futures.ThreadPoolExecutor(max_workers=VERIFIER_BATCH_WORKERS) as ex:
            futs = {ex.submit(_process_verifier_batch, b_idx, b_df): b_idx for b_idx, b_df in v_batches}
            for fut in concurrent.futures.as_completed(futs):
                result_map = fut.result()
                for ridx, out in result_map.items():
                    if out is None:
                        work_df.at[ridx, "verifier_note"] = "verifier unavailable — original verdict kept"
                        continue  # never cached — eligible for retry next run
                    verified   = bool(out.get("verified", False))
                    ambiguous  = bool(out.get("ambiguous", False))
                    note       = str(out.get("note", ""))[:150]
                    work_df.at[ridx, "verified"]           = verified
                    work_df.at[ridx, "verifier_ambiguous"] = ambiguous
                    work_df.at[ridx, "verifier_note"]      = note
                    work_df.at[ridx, "verifier_model"]     = VERIFIER_MODEL

                    # ★ CHECKPOINT: persist immediately
                    key = work_df.at[ridx, "_verifier_cache_key"]
                    with _verifier_cache_lock:
                        _verifier_cache[key] = {
                            "verified": verified,
                            "ambiguous": ambiguous,
                            "note": note,
                        }
                done_batches += 1
                # ★ CHECKPOINT: flush after every batch
                with _verifier_cache_lock:
                    _atomic_write_json(VERIFIER_CACHE_PATH, _verifier_cache)
                if done_batches % 10 == 0 or done_batches == len(v_batches):
                    elapsed = time.time() - t0
                    eta = (elapsed / done_batches) * (len(v_batches) - done_batches)
                    mismatch_note = f", {_verifier_mismatch_count} mismatch retr" \
                                    f"{'y' if _verifier_mismatch_count == 1 else 'ies'} so far" \
                                    if _verifier_mismatch_count else ""
                    print(f"  … {done_batches}/{len(v_batches)} verifier batches done "
                          f"({elapsed:.0f}s elapsed, ~{eta:.0f}s left{mismatch_note}) — "
                          f"checkpoint saved → {VERIFIER_CACHE_PATH}")
    else:
        print("  Nothing to send — every assignment was already in the checkpoint cache.")

    if _verifier_mismatch_count:
        print(f"  ⚠ Stage 4 total: {_verifier_mismatch_count} batch attempt(s) returned the "
              f"wrong item count and had to retry (set VERBOSE_BATCH_DEBUG=True in CELL 0 to "
              f"see the raw content for these next time)")

    n_disputed = int((work_df["verified"] == False).sum())
    print(f"  Verifier result: {n_disputed:,} assignment(s) disputed / flagged ambiguous")
else:
    print("Stage 4 — verifier disabled (RUN_VERIFIER_STAGE=False)")


# ============================================================================
# %% CELL 8 — MERGE + WRITE OUTPUT  (unchanged logic; two new summary rows)
# ============================================================================

bc_df = new_df.loc[barcode_matched_mask].copy()
if len(bc_df):
    ref_lookup = bc_df["_barcode_norm"].map(barcode_to_ref)
    bc_df["lulu_category_final"] = ref_lookup.apply(lambda d: d["lulu_category"] if d else "UNKNOWN")
    bc_df["category_method"]     = "barcode"
    bc_df["category_conf"]       = 1.0
    bc_df["llm_reason"]          = "direct barcode match — skipped embedding/LLM"
    bc_df["verified"]            = None
    bc_df["verifier_ambiguous"]  = None
    bc_df["verifier_note"]       = "n/a — barcode match, not sent to verifier"
    bc_df["verifier_model"]      = ""
    ref_brand = ref_lookup.apply(lambda d: d.get("brand") if d else None)
    need_brand_mask = bc_df["brand_final"].isna() | (bc_df["brand_final"].astype(str).str.strip() == "")
    bc_df.loc[need_brand_mask, "brand_final"]  = ref_brand[need_brand_mask]
    bc_df.loc[need_brand_mask, "brand_method"] = "barcode"

combined = pd.concat([bc_df, work_df], axis=0).sort_index()

combined["needs_review"] = (
    (combined["lulu_category_final"] == "UNKNOWN") |
    (combined["lulu_category_final"].isna()) |
    (combined["brand_final"].isna() | (combined["brand_final"].astype(str).str.strip() == "")) |
    (combined["verified"] == False) |
    (combined["verifier_ambiguous"] == True)
)

result_cols = ["lulu_category_final", "brand_final", "category_method", "category_conf",
               "brand_method", "llm_reason", "matched_via_barcode",
               "verified", "verifier_ambiguous", "verifier_note", "verifier_model",
               "needs_review"]
final_df = combined[ORIGINAL_COLUMNS + result_cols].rename(columns={
    "lulu_category_final": "lulu_category",
    "brand_final": "brand",
    "category_conf": "similarity_confidence_score",
})

print(f"\nWriting → {OUTPUT_XLSX}")
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="Categorized", index=False)
    final_df[final_df["needs_review"]].to_excel(writer, sheet_name="Needs Review", index=False)

    summary_rows = [
        {"Metric": "Total rows",                          "Value": len(final_df)},
        {"Metric": "Embedding model used",                "Value": EMBED_MODEL},
        {"Metric": "LLM model used",                      "Value": LLM_MODEL},
        {"Metric": "Reference corpus size (source_customer filter)", "Value": len(ref_df)},
        {"Metric": "Reference corpus source_customer filter",     "Value": str(REF_ALLOWED_SOURCE_CUSTOMERS)},
        {"Metric": "Full taxonomy category count",                "Value": len(VALID_CATEGORIES)},
        {"Metric": "Categories reachable via reference corpus",   "Value": len(_ref_only_categories)},
        {"Metric": "GPUs detected / used",                "Value": f"{_GPU_COUNT} ({', '.join(OLLAMA_HOSTS)})"},
        {"Metric": "Matched directly via Barcode",        "Value": int((final_df['category_method']=='barcode').sum())},
        {"Metric": "Category via embedding kNN (auto)",   "Value": int((final_df['category_method']=='embed_knn_auto').sum())},
        {"Metric": "Category via LLM",                    "Value": int((final_df['category_method']=='llm').sum())},
        {"Metric": "Category = UNKNOWN",                  "Value": int((final_df['lulu_category']=='UNKNOWN').sum())},
        {"Metric": "LLM thinking mode (qwen3)",            "Value": f"disabled (think={LLM_THINK})"},
        {"Metric": "Verifier model used",                 "Value": VERIFIER_MODEL if RUN_VERIFIER_STAGE else "disabled"},
        {"Metric": "Verifier thinking level (gpt-oss)",     "Value": VERIFIER_THINK_LEVEL if RUN_VERIFIER_STAGE else "n/a"},
        {"Metric": "Assignments sent to verifier",         "Value": int(final_df['verifier_model'].astype(str).ne("").sum())},
        {"Metric": "Verifier: confirmed",                   "Value": int((final_df['verified']==True).sum())},
        {"Metric": "Verifier: disputed/rejected",           "Value": int((final_df['verified']==False).sum())},
        {"Metric": "Verifier: flagged ambiguous",           "Value": int((final_df['verifier_ambiguous']==True).sum())},
        {"Metric": "Stage3 checkpoint cache size (rows)",   "Value": len(_llm_cache)},
        {"Metric": "Stage4 checkpoint cache size (rows)",   "Value": len(_verifier_cache)},
        {"Metric": "Brand via Barcode",                    "Value": int((final_df['brand_method']=='barcode').sum())},
        {"Metric": "Brand given already",                 "Value": int((final_df['brand_method']=='given').sum())},
        {"Metric": "Brand via rule (exact)",               "Value": int((final_df['brand_method']=='rule_exact').sum())},
        {"Metric": "Brand via rule (fuzzy)",               "Value": int((final_df['brand_method']=='rule_fuzzy').sum())},
        {"Metric": "Brand via LLM",                        "Value": int((final_df['brand_method']=='llm').sum())},
        {"Metric": "Brand still blank",                    "Value": int((final_df['brand'].isna() | (final_df['brand'].astype(str).str.strip()=='')).sum())},
        {"Metric": "Rows flagged needs_review",            "Value": int(final_df['needs_review'].sum())},
    ]
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

print("✓ Done")
print(pd.DataFrame(summary_rows).to_string(index=False))
print(f"\nTo download just the result (not a zip): open the Output tab, find "
      f"'output/{os.path.basename(OUTPUT_XLSX)}', and click its individual download icon. "
      f"Only the 'Download All' button zips every file in /kaggle/working together "
      f"(cache/, logs, etc.) — downloading this one file by itself gives you the .xlsx directly.")